# DNA Codec v3.0 — Official Benchmark Suite
**Author:** Francisco Molina-Burgos · ORCID 0009-0008-6093-8267  
**Reference environment:** Google Colab Standard T4 GPU  
**Purpose:** Reproducible benchmark figures for article submission.

---
**INSTRUCTIONS**
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**
3. The final cell prints the full benchmark report — copy it into the article.

In [ ]:
# ── CELL 1: Hardware fingerprint ─────────────────────────────────────────────
import platform, psutil, subprocess, sys, os

def _cpu_model():
    try:
        with open('/proc/cpuinfo') as f:
            for line in f:
                if 'model name' in line:
                    return line.split(':', 1)[1].strip()
    except: pass
    return platform.processor() or 'Unknown'

def _gpu_name():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                            '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=5)
        line = r.stdout.strip()
        if line:
            parts = line.split(',')
            return parts[0].strip(), parts[1].strip() if len(parts) > 1 else ''
    except: pass
    return 'N/A (CPU-only)', ''

gpu_name, gpu_vram = _gpu_name()
HW = {
    'cpu'   : _cpu_model(),
    'gpu'   : gpu_name,
    'vram'  : gpu_vram,
    'ram_gb': round(psutil.virtual_memory().total / 1e9, 1),
    'python': platform.python_version(),
}
for k, v in HW.items():
    print(f'  {k:<8}: {v}')
print('Hardware fingerprint recorded ✓')

In [ ]:
# ── CELL 2: Install dependencies ─────────────────────────────────────────────
!pip install reedsolo cryptography beautifulsoup4 requests -q
print('Dependencies installed ✓')

In [ ]:
# ── CELL 3: Clone repository ──────────────────────────────────────────────────
!git clone --depth 1 https://github.com/Yatrogenesis/DNA-Codec-Advanced.git /content/dna_codec -q
sys.path.insert(0, '/content/dna_codec')
from dna_codec_v3 import ContainerDNACodec, BiologicalSequencingSimulator
from reedsolo import RSCodec
print('Repository cloned and codec imported ✓')
print(f'RSCodec available: {RSCodec}')

In [ ]:
# ── CELL 4: Shared utilities ──────────────────────────────────────────────────
import numpy as np, random, time, statistics

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

def ci95(data):
    n = len(data)
    return 1.96 * statistics.stdev(data) / n**0.5 if n > 1 else 0

RESULTS = {}   # populated by experiment cells
print('Utilities ready ✓  (seed=42)')

In [ ]:
# ── CELL 5: EXPERIMENT 1 ─ Ambiguity Robustness  (n=100)
# Payloads intentionally contain both boundary marker sequences as literal text.
# Validates that the explicit LENGTH field prevents any parsing ambiguity.
np.random.seed(SEED); random.seed(SEED)

codec = ContainerDNACodec()
N, ok, times_ms = 100, 0, []

for i in range(N):
    payload = (
        f'Replicate {i}: the sequence GATTACACAGTA (START) and '
        'TCATGTACAATC (END) appear as literal payload text, '
        'along with type codes AAAA AAAT AATA AATG AATC.'
    )
    t0 = time.perf_counter()
    enc = codec.encode(payload)
    dec, meta = codec.decode(enc)
    elapsed = (time.perf_counter() - t0) * 1000
    if dec.decode('utf-8') == payload and meta['checksum_valid']:
        ok += 1
        times_ms.append(elapsed)

RESULTS['exp1'] = {
    'n': N, 'success': ok,
    'success_pct': ok / N * 100,
    'mean_time_ms': statistics.mean(times_ms) if times_ms else 0,
    'ci95_ms': ci95(times_ms),
}
print(f'Exp 1 done  — {ok}/{N} ({ok/N*100:.1f}%) ✓')

In [ ]:
# ── CELL 6: EXPERIMENT 2 ─ Reed-Solomon Error Correction  (n=100)
# Encodes with RS enabled, applies Nanopore-profile sequencing errors
# (2% substitution + 1.5% indel — the harshest platform model),
# then measures how often RS restores byte-exact fidelity.
np.random.seed(SEED); random.seed(SEED)

codec_rs  = ContainerDNACodec(enable_reed_solomon=True)
sim_nano  = BiologicalSequencingSimulator(platform='Nanopore')
payload   = b'DNA Codec v3.0 Reed-Solomon resilience test. ' * 40

N, ok, detected = 100, 0, 0

for seed_i in range(N):
    encoded = codec_rs.encode(payload)
    # Apply realistic Nanopore error profile
    noisy, err_stats = sim_nano.simulate_errors(encoded, seed=seed_i)
    try:
        dec, meta = codec_rs.decode(noisy)
        if meta['checksum_valid'] and dec == payload:
            ok += 1
        else:
            detected += 1   # decoded but checksum mismatch = detected, not recovered
    except Exception:
        detected += 1

RESULTS['exp2'] = {
    'n': N, 'success': ok, 'detected': detected,
    'success_pct': ok / N * 100,
    'detected_pct': detected / N * 100,
    'platform': 'Nanopore (2% sub + 1.5% indel)',
}
print(f'Exp 2 done  — recovered {ok}/{N} ({ok/N*100:.1f}%),  '
      f'detected-not-recovered {detected}/{N} ✓')

In [ ]:
# ── CELL 7: EXPERIMENT 3 ─ Throughput Benchmark  (n=20)
# 50,000-nucleotide equivalent payload (12,500 bytes binary).
np.random.seed(SEED); random.seed(SEED)

codec_b   = ContainerDNACodec()
payload_b = bytes(random.getrandbits(8) for _ in range(12500))

N, enc_t, dec_t, nt_len = 20, [], [], None

for _ in range(N):
    t0 = time.perf_counter()
    dna = codec_b.encode(payload_b)
    enc_t.append(time.perf_counter() - t0)
    if nt_len is None:
        nt_len = len(dna)
    t0 = time.perf_counter()
    codec_b.decode(dna)
    dec_t.append(time.perf_counter() - t0)

mean_enc   = statistics.mean(enc_t)
throughput = (nt_len / 1e6) / mean_enc
density    = len(payload_b) / nt_len

RESULTS['exp3'] = {
    'n': N,
    'payload_bytes': len(payload_b),
    'encoded_nt': nt_len,
    'mean_encode_ms': mean_enc * 1000,
    'ci95_encode_ms': ci95(enc_t) * 1000,
    'mean_decode_ms': statistics.mean(dec_t) * 1000,
    'throughput_mbps': throughput,
    'density_bytes_per_nt': density,
    'theoretical_max_pct': density / 0.25 * 100,
}
print(f'Exp 3 done  — {throughput:.3f} Mbp/s  |  {density:.4f} bytes/nt ✓')

In [ ]:
# ── CELL 8: EXPERIMENT 4 ─ Per-platform error resilience  (n=30 each)
# Applies each platform's error profile without RS.
# Measures base checksum integrity survival rate.
np.random.seed(SEED); random.seed(SEED)

codec_e   = ContainerDNACodec()
payload_e = (b'DNA Codec v3.0 platform resilience test. ' * 30)
encoded_e = codec_e.encode(payload_e)

PLATFORMS = ['Illumina', 'Nanopore', 'PacBio', 'Synthesis']
N = 30
plat_res = {}

for plat in PLATFORMS:
    sim = BiologicalSequencingSimulator(platform=plat)
    ok  = 0
    for seed_i in range(N):
        noisy, _stats = sim.simulate_errors(encoded_e, seed=seed_i)
        try:
            _, meta = codec_e.decode(noisy)
            if meta['checksum_valid']:
                ok += 1
        except Exception:
            pass
    plat_res[plat] = {'n': N, 'success': ok, 'pct': ok / N * 100}
    print(f'  {plat:<12}: {ok}/{N}  ({ok/N*100:.0f}%)')

RESULTS['exp4'] = plat_res
print('Exp 4 done ✓')

In [ ]:
# ── CELL 9: FULL BENCHMARK REPORT  (copy into article) ───────────────────────
import datetime

W = 62
BOX  = '║'
SEP  = '─' * (W - 2)
DSEP = '═' * (W - 2)

def hdr(text):
    print('╠' + DSEP + '╣')
    print(f'{BOX}{text:^{W-2}}{BOX}')

def sub():
    print('╠' + SEP + '╣')

def row(label, value):
    lbl = f'  {label}'
    print(f'{BOX}{lbl:<34}{value:<{W-36}}{BOX}')

def blank():
    print(f'{BOX}{" "*(W-2)}{BOX}')

print('╔' + DSEP + '╗')
print(f'{BOX}{"DNA CODEC v3.0 — BENCHMARK REPORT":^{W-2}}{BOX}')
ts = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M UTC')
print(f'{BOX}{ts:^{W-2}}{BOX}')

hdr('HARDWARE')
sub()
row('Environment:', 'Google Colab Standard T4 GPU')
row('CPU:', HW['cpu'][:36])
row('GPU:', HW['gpu'])
row('GPU VRAM:', HW['vram'])
row('RAM:', f'{HW["ram_gb"]} GB')
row('Python:', HW['python'])
row('Seed (all experiments):', str(SEED))

hdr('EXPERIMENT RESULTS')

# Exp 1
sub()
e1 = RESULTS['exp1']
row('Exp.1  Ambiguity Robustness', f'n={e1["n"]}')
row('  Payload:', 'Contains both marker sequences as literal text')
row('  Success rate:', f'{e1["success_pct"]:.1f}%  ({e1["success"]}/{e1["n"]})')
row('  Mean round-trip time:', f'{e1["mean_time_ms"]:.2f} ms  (95% CI ±{e1["ci95_ms"]:.2f} ms)')

# Exp 2
sub()
e2 = RESULTS['exp2']
row('Exp.2  Reed-Solomon Error Correction', f'n={e2["n"]}')
row('  Error profile:', e2['platform'])
row('  RS corrected + exact recovery:', f'{e2["success_pct"]:.1f}%  ({e2["success"]}/{e2["n"]})')
row('  Detected (not recovered):', f'{e2["detected_pct"]:.1f}%  ({e2["detected"]}/{e2["n"]})')

# Exp 3
sub()
e3 = RESULTS['exp3']
row('Exp.3  Throughput Benchmark', f'n={e3["n"]}')
row('  Payload:', f'{e3["payload_bytes"]:,} bytes  →  {e3["encoded_nt"]:,} nt')
row('  Encoding time:', f'{e3["mean_encode_ms"]:.1f} ms  (95% CI ±{e3["ci95_encode_ms"]:.1f} ms)')
row('  Decoding time:', f'{e3["mean_decode_ms"]:.1f} ms')
row('  Throughput:', f'{e3["throughput_mbps"]:.3f} Mbp/s')
row('  Storage density:', f'{e3["density_bytes_per_nt"]:.4f} bytes/nt  ({e3["theoretical_max_pct"]:.1f}% of 0.25 max)')

# Exp 4
sub()
e4 = RESULTS.get('exp4', {})
row('Exp.4  Per-platform Checksum Survival', f'n=30 each')
if e4:
    for p, r in e4.items():
        row(f'  {p}:', f'{r["pct"]:.0f}%  ({r["success"]}/{r["n"]})')
else:
    row('  Result:', 'EXPERIMENT FAILED — check Cell 8')

print('╚' + DSEP + '╝')
print()
print('Copy the report above into the article supplementary material.')
print('Primary values for article body:')
print(f'  Exp.1 success   → {RESULTS["exp1"]["success_pct"]:.1f}%')
print(f'  Exp.2 RS corr.  → {RESULTS["exp2"]["success_pct"]:.1f}%')
print(f'  Exp.3 throughput→ {RESULTS["exp3"]["throughput_mbps"]:.3f} Mbp/s')
print(f'  Exp.3 density   → {RESULTS["exp3"]["density_bytes_per_nt"]:.4f} bytes/nt')